### Импорты и пути

In [1]:
import sys
sys.path.append('../src')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import importlib
import time

from tqdm.auto import tqdm

import rl_train as rl_train_mod
importlib.reload(rl_train_mod)
from rl_train import train_rl

import judge as judge_mod
importlib.reload(judge_mod)
from judge import JudgeModel

import reward_functions as rf_mod
importlib.reload(rf_mod)
from reward_functions import (
    RMReward, ItemCenteredReward, build_item_mean_map,
)

from dataset import build_item_vocab
from model import create_model
from reward_model import create_reward_model
from rm_index import RMIndex
from my_evaluate import GPT2Predictor, evaluate_gpt2_fast
from metrics import (
    build_eval_targets, build_context,
    recall_at_k, ndcg_at_k, average_precision_at_k,
)

PROC = Path('../data/processed')
RESULTS = Path('../results')
RESULTS_R2 = RESULTS / 'rl_r2_results'
RESULTS_R2.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


### Данные и словарь

In [2]:
train_df = pd.read_parquet(PROC / 'train.parquet')
val_df   = pd.read_parquet(PROC / 'val.parquet')
test_df  = pd.read_parquet(PROC / 'test.parquet')

item2idx, idx2item = build_item_vocab(train_df)
vocab_size = len(item2idx) + 2
print(f"Vocab size: {vocab_size}")

Vocab size: 14937


###  π_ref и RM

In [3]:
def load_gpt2_policy(ckpt):
    m = create_model(vocab_size=vocab_size)
    m.load_state_dict(torch.load(ckpt, map_location='cpu'))
    return m

ref_policy = load_gpt2_policy(RESULTS / 'checkpoints_v2' / 'best_model.pt').to(device).eval()
for p in ref_policy.parameters():
    p.requires_grad = False

rm = create_reward_model(vocab_size=vocab_size,
                         init_from=RESULTS / 'checkpoints_v2' / 'best_model.pt')
rm.load_state_dict(torch.load(RESULTS / 'checkpoints_rm_v2' / 'best_model.pt',
                              map_location='cpu'))
rm = rm.to(device).eval()
for p in rm.parameters():
    p.requires_grad = False
print(f"RM: {sum(p.numel() for p in rm.parameters()):,} params, frozen")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 14936), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 14936), got 50256. This may result in unexpected behavior.


[create_reward_model] Backbone: загружено 28 тензоров
[create_reward_model] candidate_emb инициализирован из wte
RM: 4,268,289 params, frozen


### RMIndex

In [4]:
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
idx = RMIndex(all_df, item2idx, t1='2021-07-24', t2='2021-08-04')

RMIndex: 4,632,197 событий
  train: 2,789,962
  val:   371,021
  test:  474,130


### Helper-функции (score_candidates, _batched_policy_logits, fixed_eval)

In [5]:
@torch.no_grad()
def _batched_policy_logits(policy, contexts, item2idx, max_len=50, device='cuda',
                            batch_b=512, apply_masking=True, desc=None):
    B = len(contexts)
    out = []
    n_batches = (B + batch_b - 1) // batch_b
    pbar = tqdm(range(0, B, batch_b),
                desc=desc or "policy logits",
                total=n_batches, leave=False)
    for s in pbar:
        end = min(s + batch_b, B)
        inp = torch.zeros(end - s, max_len, dtype=torch.long, device=device)
        for i, ctx in enumerate(contexts[s:end]):
            recent = ctx[-max_len:]
            indexed = [item2idx.get(int(x), 1) for x in recent]
            if indexed:
                inp[i, -len(indexed):] = torch.tensor(indexed, device=device)
        try:
            logits = policy(inp, logits_to_keep=1).logits[:, -1, :].clone()
        except TypeError:
            logits = policy(inp).logits[:, -1, :].clone()
        if apply_masking:
            logits[:, 0] = -1e9
            logits[:, 1] = -1e9
            for i, ctx in enumerate(contexts[s:end]):
                s_items = [item2idx[int(x)] for x in ctx if int(x) in item2idx]
                if s_items:
                    logits[i, torch.tensor(s_items, device=device)] = -1e9
        out.append(logits)
    return torch.cat(out, dim=0)


@torch.no_grad()
def score_candidates(rm, contexts, cand_idx, item2idx,
                     max_len=50, chunk=8192, batch_b=256, device='cuda', desc=None):
    B = len(contexts)
    L = max_len
    M = cand_idx.shape[1]

    hist = torch.zeros(B, L - 1, dtype=torch.long, device=device)
    for i, ctx in enumerate(contexts):
        recent = ctx[-(L - 1):] if ctx else []
        indexed = [item2idx.get(int(x), 1) for x in recent]
        if indexed:
            hist[i, -len(indexed):] = torch.tensor(indexed, device=device)

    out = []
    n_batches = (B + batch_b - 1) // batch_b
    pbar = tqdm(range(0, B, batch_b),
                desc=desc or f"RM score {B}x{M}",
                total=n_batches, leave=False)
    for s in pbar:
        end = min(s + batch_b, B)
        h = hist[s:end].unsqueeze(1).expand(-1, M, -1)
        c = cand_idx[s:end].unsqueeze(2)
        seq = torch.cat([h, c], dim=2).reshape(-1, L)
        for s2 in range(0, seq.size(0), chunk):
            out.append(torch.sigmoid(rm(seq[s2:s2 + chunk])))
    return torch.cat(out).view(B, M)


@torch.no_grad()
def fixed_eval(policy, ref_policy, rm, user_ids, contexts, user_targets,
               item2idx, idx2item, K=10, max_len=50, device='cuda'):
    logits_pi  = _batched_policy_logits(policy,     contexts, item2idx, max_len, device)
    logits_ref = _batched_policy_logits(ref_policy, contexts, item2idx, max_len, device)

    logp_pi  = torch.log_softmax(logits_pi,  dim=-1)
    logp_ref = torch.log_softmax(logits_ref, dim=-1)
    kl = (logp_pi.exp() * (logp_pi - logp_ref)).sum(dim=-1).mean().item()

    topK = torch.topk(logits_pi, K, dim=-1).indices
    rm_scores = score_candidates(rm, contexts, topK, item2idx)
    mean_rm_top10 = rm_scores.mean().item()

    topK_lists = [[idx2item[int(i)] for i in row] for row in topK.cpu()]
    ndcg_v, recall_v, map_v = [], [], []
    for i, uid in enumerate(user_ids):
        targets = user_targets.get(uid, [])
        if not targets:
            continue
        preds = topK_lists[i]
        for t in targets:
            ndcg_v.append(ndcg_at_k(preds, t, K))
            recall_v.append(recall_at_k(preds, t, K))
            map_v.append(average_precision_at_k(preds, t, K))

    return dict(
        kl=kl, mean_rm_top10=mean_rm_top10,
        ndcg10=float(np.mean(ndcg_v)) if ndcg_v else 0.0,
        recall10=float(np.mean(recall_v)) if recall_v else 0.0,
        map10=float(np.mean(map_v)) if map_v else 0.0,
        n_targets=len(ndcg_v),
    )

### Контексты для fixed_eval (2000 test)

In [ ]:
test_targets = build_eval_targets(test_df)
test_context = build_context(train_df, val_df)

test_users = test_targets['user_id'].unique()
np.random.seed(123)
eval_users = np.random.choice(test_users, size=2000, replace=False).tolist()

eval_contexts = []
for u in eval_users:
    ctx = test_context.get(u)
    items = ctx['items'] if ctx else []
    eval_contexts.append(list(items)[-50:])

user_targets = {}
for u in eval_users:
    user_targets[u] = test_targets[test_targets['user_id'] == u]['item_id'].tolist()

n_targets = sum(len(v) for v in user_targets.values())
print(f"Eval: {len(eval_users)} пользователей, {n_targets} таргетов")
assert all(len(c) > 0 for c in eval_contexts), "Пустой контекст - фильтр холодных должен был исключить"

Eval: 2000 пользователей, 2767 таргетов


### Base (π_ref) fixed_eval

In [7]:
print("[base] fixed_eval на π_ref ...")
base_fixed = fixed_eval(
    policy=ref_policy, ref_policy=ref_policy, rm=rm,
    user_ids=eval_users, contexts=eval_contexts, user_targets=user_targets,
    item2idx=item2idx, idx2item=idx2item,
)
print(f"[base] Er={base_fixed['mean_rm_top10']:.4f}  "
      f"NDCG@10={base_fixed['ndcg10']:.4f}  Recall@10={base_fixed['recall10']:.4f}")

[base] fixed_eval на π_ref ...


policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

RM score 2000x10:   0%|          | 0/8 [00:00<?, ?it/s]

[base] Er=0.5096  NDCG@10=0.1361  Recall@10=0.2255


### Построение item_mean_map и ItemCenteredReward

In [10]:
item_mean_map = build_item_mean_map(idx, idx2item)
print(f"[R2] item_mean_map: {len(item_mean_map)} айтемов")
print(f"[R2] Пример: item {list(item_mean_map.keys())[0]} -> mean={list(item_mean_map.values())[0]:.4f}")

rm_reward = RMReward(rm=rm, item2idx=item2idx, max_len=50, device=device)
item_centered_reward = ItemCenteredReward(
    rm=rm,
    item2idx=item2idx,
    item_mean_dict=item_mean_map,
    default_mean=0.5,
    max_len=50,
    device=device,
)

# Проверка на 5 примерах: R2 = R0 - item_mean
sample_rows = np.random.choice(np.flatnonzero(idx.train_mask), size=5, replace=False)
ctxs, acts = [], []
for i in sample_rows:
    hi = idx.day_start[i]
    lo = max(idx.user_start[i], hi - 49)
    ctx = [idx2item[int(idx.items[j])] for j in range(lo, hi)]
    ctxs.append(ctx)
    acts.append(idx2item[int(idx.items[i])])

r0_vals = rm_reward(ctxs, acts).cpu()
r2_vals = item_centered_reward(ctxs, acts).cpu()
print("\nПроверка R2 = R0 - item_mean:")
for a, r0, r2 in zip(acts, r0_vals, r2_vals):
    im = item_mean_map.get(a, 0.5)
    diff = (r0.item() - im) - r2.item()
    print(f"  item {a}: R0={r0:.4f}  item_mean={im:.4f}  R0-im={r0.item()-im:+.4f}  "
          f"R2={r2:+.4f}  |diff|={abs(diff):.2e}")

assert all(abs(((r0.item() - item_mean_map.get(a, 0.5)) - r2.item())) < 1e-5
           for a, r0, r2 in zip(acts, r0_vals, r2_vals)), "R2 != R0 - item_mean!"
print("ItemCenteredReward эквивалентна R0 - item_mean")

[R2] item_mean_map: 14434 айтемов
[R2] Пример: item 0 -> mean=0.0980

Проверка R2 = R0 - item_mean:
  item 14703: R0=0.4982  item_mean=0.5464  R0-im=-0.0482  R2=-0.0482  |diff|=0.00e+00
  item 10440: R0=0.3278  item_mean=0.5717  R0-im=-0.2439  R2=-0.2439  |diff|=0.00e+00
  item 10837: R0=0.1853  item_mean=0.1801  R0-im=+0.0052  R2=+0.0052  |diff|=0.00e+00
  item 5732: R0=0.4300  item_mean=0.4163  R0-im=+0.0137  R2=+0.0137  |diff|=0.00e+00
  item 3076: R0=0.2060  item_mean=0.2436  R0-im=-0.0376  R2=-0.0376  |diff|=0.00e+00
ItemCenteredReward эквивалентна R0 - item_mean


### Запуск RL R2

In [11]:
betas_r2 = [0.5, 1.0, 2.0]
n_steps_r2 = 500
lr_r2 = 1e-5

r2_results = {}
histories_r2 = {}

for beta in betas_r2:
    print(f"\n{'='*70}\nR2 (Item-Centered) с β={beta}\n{'='*70}")
    policy_r2 = load_gpt2_policy(RESULTS / 'checkpoints_v2' / 'best_model.pt').to(device)

    t0 = time.time()
    history_r2 = train_rl(
        policy=policy_r2, ref_policy=ref_policy,
        reward_fn=item_centered_reward, rm_index=idx,
        idx2item=idx2item, item2idx=item2idx,
        n_steps=n_steps_r2, batch_size=64,
        lr=lr_r2, beta_kl=beta, G=8, top_m=200, max_len=50,
        device=device,
        save_dir=str(RESULTS / f'checkpoints_rl_r2_beta{beta}'),
        seed=1, log_every=100, warmup_steps=50,
    )
    histories_r2[beta] = history_r2
    print(f"Обучение: {time.time()-t0:.0f}s")

    metrics_r2 = fixed_eval(
        policy=policy_r2, ref_policy=ref_policy, rm=rm,
        user_ids=eval_users, contexts=eval_contexts, user_targets=user_targets,
        item2idx=item2idx, idx2item=idx2item,
    )
    r2_results[beta] = metrics_r2
    print(f"fixed_eval R2 β={beta}: Er={metrics_r2['mean_rm_top10']:.4f}  "
          f"KL={metrics_r2['kl']:.4f}  NDCG@10={metrics_r2['ndcg10']:.4f}  "
          f"Recall@10={metrics_r2['recall10']:.4f}")

    del policy_r2
    torch.cuda.empty_cache()

print("\nR2 завершён")


R2 (Item-Centered) с β=0.5
Train contexts available: 2,789,962

Начинаем RL: 500 шагов, batch=64, lr=1e-05, β=0.5, G=8, M=200

Step  100/500 | pg_loss=-0.3870 | kl=0.0653 | mean_r=0.0018 | |A|=0.770 | max π_θ=0.137 | 17s
Step  200/500 | pg_loss=-0.3403 | kl=0.0846 | mean_r=0.0151 | |A|=0.766 | max π_θ=0.169 | 33s
Step  300/500 | pg_loss=-0.5671 | kl=0.1092 | mean_r=0.0128 | |A|=0.761 | max π_θ=0.196 | 50s
Step  400/500 | pg_loss=-0.3545 | kl=0.1138 | mean_r=-0.0006 | |A|=0.774 | max π_θ=0.186 | 66s
Step  500/500 | pg_loss=-0.4500 | kl=0.1386 | mean_r=-0.0016 | |A|=0.753 | max π_θ=0.182 | 82s

RL завершено: 500 шагов за 82s
Финальные метрики (последние 20 шагов):
  mean_reward = 0.0111
  kl          = 0.1288
Чекпоинты в: ..\results\checkpoints_rl_r2_beta0.5
Обучение: 82s


policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

RM score 2000x10:   0%|          | 0/8 [00:00<?, ?it/s]

fixed_eval R2 β=0.5: Er=0.4947  KL=0.1567  NDCG@10=0.1338  Recall@10=0.2212

R2 (Item-Centered) с β=1.0
Train contexts available: 2,789,962

Начинаем RL: 500 шагов, batch=64, lr=1e-05, β=1.0, G=8, M=200

Step  100/500 | pg_loss=-0.3377 | kl=0.0631 | mean_r=0.0038 | |A|=0.773 | max π_θ=0.136 | 16s
Step  200/500 | pg_loss=-0.3431 | kl=0.0729 | mean_r=0.0147 | |A|=0.770 | max π_θ=0.160 | 33s
Step  300/500 | pg_loss=-0.5286 | kl=0.0835 | mean_r=0.0115 | |A|=0.760 | max π_θ=0.179 | 49s
Step  400/500 | pg_loss=-0.4363 | kl=0.0798 | mean_r=-0.0035 | |A|=0.773 | max π_θ=0.162 | 65s
Step  500/500 | pg_loss=-0.3868 | kl=0.0978 | mean_r=-0.0016 | |A|=0.766 | max π_θ=0.160 | 83s

RL завершено: 500 шагов за 83s
Финальные метрики (последние 20 шагов):
  mean_reward = 0.0088
  kl          = 0.0911
Чекпоинты в: ..\results\checkpoints_rl_r2_beta1.0
Обучение: 83s


...

fixed_eval R2 β=1.0: Er=0.4984  KL=0.1125  NDCG@10=0.1336  Recall@10=0.2205

R2 (Item-Centered) с β=2.0
Train contexts available: 2,789,962

Начинаем RL: 500 шагов, batch=64, lr=1e-05, β=2.0, G=8, M=200

Step  100/500 | pg_loss=-0.3858 | kl=0.0594 | mean_r=0.0027 | |A|=0.773 | max π_θ=0.132 | 17s
Step  200/500 | pg_loss=-0.2854 | kl=0.0599 | mean_r=0.0166 | |A|=0.769 | max π_θ=0.148 | 34s
Step  300/500 | pg_loss=-0.5117 | kl=0.0606 | mean_r=0.0099 | |A|=0.777 | max π_θ=0.159 | 53s
Step  400/500 | pg_loss=-0.3726 | kl=0.0551 | mean_r=-0.0058 | |A|=0.771 | max π_θ=0.141 | 70s
Step  500/500 | pg_loss=-0.2888 | kl=0.0684 | mean_r=-0.0031 | |A|=0.765 | max π_θ=0.139 | 88s

RL завершено: 500 шагов за 88s
Финальные метрики (последние 20 шагов):
  mean_reward = 0.0058
  kl          = 0.0628
Чекпоинты в: ..\results\checkpoints_rl_r2_beta2.0
Обучение: 88s


policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

RM score 2000x10:   0%|          | 0/8 [00:00<?, ?it/s]

fixed_eval R2 β=2.0: Er=0.5025  KL=0.0799  NDCG@10=0.1348  Recall@10=0.2226

R2 завершён


### Лучшая β для R2

In [12]:
valid_r2 = {b: m for b, m in r2_results.items()
            if m['ndcg10'] >= 0.97 * base_fixed['ndcg10']}

if valid_r2:
    best_beta_r2 = max(valid_r2, key=lambda b: valid_r2[b]['mean_rm_top10'])
    print(f"Победитель R2: β={best_beta_r2}")
    print(f"Er={valid_r2[best_beta_r2]['mean_rm_top10']:.4f}  "
          f"NDCG@10={valid_r2[best_beta_r2]['ndcg10']:.4f}")
else:
    best_beta_r2 = min(r2_results, key=lambda b: abs(r2_results[b]['ndcg10'] - 0.97 * base_fixed['ndcg10']))
    print(f"Ни одна β не прошла порог NDCG ≥ 97% base.")
    print(f"Берём β={best_beta_r2} как минимальную потерю (NDCG={r2_results[best_beta_r2]['ndcg10']:.4f})")

print(f"\nВсе конфигурации R2:")
print(f"{'β':>8}  {'Er':>7}  {'KL':>7}  {'NDCG@10':>8}  {'Recall@10':>10}")
print("-" * 50)
for beta, m in r2_results.items():
    print(f"{beta:>8.1f}  {m['mean_rm_top10']:>7.4f}  {m['kl']:>7.4f}  "
          f"{m['ndcg10']:>8.4f}  {m['recall10']:>10.4f}")

Победитель R2: β=2.0
Er=0.5025  NDCG@10=0.1348

Все конфигурации R2:
       β       Er       KL   NDCG@10   Recall@10
--------------------------------------------------
     0.5   0.4947   0.1567    0.1338      0.2212
     1.0   0.4984   0.1125    0.1336      0.2205
     2.0   0.5025   0.0799    0.1348      0.2226


### Сравнение R2 с base на полном test

In [13]:
def compute_metrics(preds_dict, targets_df, k_values=(10, 50, 100)):
    max_k = max(k_values)
    accum = {f'{m}@{k}': [] for k in k_values for m in ['Recall', 'NDCG', 'MAP']}
    for _, row in targets_df.iterrows():
        preds = preds_dict.get(row['user_id'], [])[:max_k]
        t = row['item_id']
        for k in k_values:
            accum[f'Recall@{k}'].append(recall_at_k(preds, t, k))
            accum[f'NDCG@{k}'].append(ndcg_at_k(preds, t, k))
            accum[f'MAP@{k}'].append(average_precision_at_k(preds, t, k))
    return {k: float(np.mean(v)) for k, v in accum.items()}

@torch.no_grad()
def mean_rm_reward_on_full_test(preds_dict, context_dict, rm, item2idx,
                                 K=10, batch_size=512, device='cuda'):
    users = list(preds_dict.keys())
    ctxs = [context_dict.get(u, {}).get('items', []) for u in users]
    cand = torch.zeros(len(users), K, dtype=torch.long, device=device)
    for i, u in enumerate(users):
        preds = preds_dict[u][:K]
        for j, item in enumerate(preds):
            cand[i, j] = item2idx.get(int(item), 1)
    rewards = score_candidates(rm, ctxs, cand, item2idx,
                               batch_b=batch_size, device=device)
    return rewards.mean().item()

print(f"[test] загружаем чекпоинт R2 β={best_beta_r2}")
policy_best_r2 = load_gpt2_policy(
    RESULTS / f'checkpoints_rl_r2_beta{best_beta_r2}' / 'final_model.pt'
).to(device).eval()

print("[test] R2 inference на полном test ...")
test_preds_r2 = evaluate_gpt2_fast(
    GPT2Predictor(policy_best_r2, item2idx, idx2item, max_len=50),
    test_targets, test_context, k=100, batch_size=512, exclude_seen=True,
)
r2_full = compute_metrics(test_preds_r2, test_targets)

print("[test] base inference на полном test ...")
test_preds_base = evaluate_gpt2_fast(
    GPT2Predictor(ref_policy, item2idx, idx2item, max_len=50),
    test_targets, test_context, k=100, batch_size=512, exclude_seen=True,
)
base_full = compute_metrics(test_preds_base, test_targets)

print(f"\n{'Метрика':<15} {'Base':>10} {'R2':>10} {'Δ':>10}")
print("=" * 50)
for m in base_full:
    d = (r2_full[m] / base_full[m] - 1) * 100 if base_full[m] > 0 else 0
    print(f"{m:<15} {base_full[m]:>10.4f} {r2_full[m]:>10.4f} {d:>+9.1f}%")

[test] загружаем чекпоинт R2 β=2.0
[test] R2 inference на полном test ...
Уникальных юзеров для оценки: 132,323


predict_batch:   0%|          | 0/259 [00:00<?, ?it/s]

[test] base inference на полном test ...
Уникальных юзеров для оценки: 132,323


predict_batch:   0%|          | 0/259 [00:00<?, ?it/s]


Метрика               Base         R2          Δ
Recall@10           0.2192     0.2188      -0.2%
NDCG@10             0.1304     0.1305      +0.0%
MAP@10              0.1030     0.1032      +0.2%
Recall@50           0.3714     0.3714      -0.0%
NDCG@50             0.1638     0.1639      +0.1%
MAP@50              0.1101     0.1103      +0.2%
Recall@100          0.4592     0.4582      -0.2%
NDCG@100            0.1780     0.1780      -0.0%
MAP@100             0.1113     0.1115      +0.2%


### RM reward на полном test

In [14]:
print("[reward] RM reward топ-10 по базе ...")
reward_base_full = mean_rm_reward_on_full_test(test_preds_base, test_context, rm, item2idx, K=10)

print("[reward] RM reward топ-10 по R2 ...")
reward_r2_full = mean_rm_reward_on_full_test(test_preds_r2, test_context, rm, item2idx, K=10)

print(f"\n{'='*60}")
print(f"RM reward топ-10 (full test)")
print(f"{'='*60}")
print(f"  Base:  {reward_base_full:.4f}")
print(f"  R2:    {reward_r2_full:.4f}")
print(f"  Δ:     {(reward_r2_full/reward_base_full-1)*100:+.1f}%")

[reward] RM reward топ-10 по базе ...


RM score 132323x10:   0%|          | 0/259 [00:00<?, ?it/s]

[reward] RM reward топ-10 по R2 ...


RM score 132323x10:   0%|          | 0/259 [00:00<?, ?it/s]


RM reward топ-10 (full test)
  Base:  0.5092
  R2:    0.5011
  Δ:     -1.6%


### Валидация судьёй

In [15]:
judge = JudgeModel.load(RESULTS / 'checkpoints_judge' / 'judge')

print("[judge] оценка base на полном test ...")
judge_base_full  = judge.score_predictions_topk(test_preds_base, K=10)
mean_item_base   = judge.mean_item_mean_topk(test_preds_base, K=10)

print("[judge] оценка R2 на полном test ...")
judge_r2_full    = judge.score_predictions_topk(test_preds_r2, K=10)
mean_item_r2     = judge.mean_item_mean_topk(test_preds_r2, K=10)

d_judge = (judge_r2_full / judge_base_full - 1) * 100
d_itemmean = (mean_item_r2 / mean_item_base - 1) * 100

print(f"\n{'='*72}")
print(f"Сравнение оценок reward (full test)")
print(f"{'='*72}")
print(f"{'Метрика':<30} {'Base':>10} {'R2':>12} {'Δ':>10}")
print("-" * 72)
print(f"{'RM reward top-10':<30} {reward_base_full:>10.4f} {reward_r2_full:>12.4f} "
      f"{(reward_r2_full/reward_base_full-1)*100:>+9.1f}%")
print(f"{'Judge reward top-10':<30} {judge_base_full:>10.4f} {judge_r2_full:>12.4f} "
      f"{d_judge:>+9.1f}%")
print(f"{'Mean item_mean top-10':<30} {mean_item_base:>10.4f} {mean_item_r2:>12.4f} "
      f"{d_itemmean:>+9.1f}%")
print("-" * 72)

[judge] оценка base на полном test ...
[judge] оценка R2 на полном test ...

Сравнение оценок reward (full test)
Метрика                              Base           R2          Δ
------------------------------------------------------------------------
RM reward top-10                   0.5092       0.5011      -1.6%
Judge reward top-10                0.4817       0.4740      -1.6%
Mean item_mean top-10              0.4850       0.4731      -2.4%
------------------------------------------------------------------------


### Сохранение результатов R2

In [16]:
# Свип R2
sweep_rows = [{'model': 'base', **base_fixed}]
for beta, m in r2_results.items():
    sweep_rows.append({'model': f'R2_beta={beta}', **m})
pd.DataFrame(sweep_rows).to_csv(RESULTS_R2 / 'fixed_eval_r2_by_beta.csv', index=False)

# Полный test
pd.DataFrame({
    'metric': ['reward_top10', 'NDCG@10', 'Recall@10', 'MAP@10'],
    'base': [reward_base_full, base_full['NDCG@10'], base_full['Recall@10'], base_full['MAP@10']],
    f'r2_beta{best_beta_r2}': [reward_r2_full, r2_full['NDCG@10'], r2_full['Recall@10'], r2_full['MAP@10']],
}).to_csv(RESULTS_R2 / 'full_test_r2.csv', index=False)

# Судья
pd.DataFrame({
    'metric': ['rm_reward_top10', 'judge_reward_top10', 'mean_item_mean_top10'],
    'base': [reward_base_full, judge_base_full, mean_item_base],
    f'r2_beta{best_beta_r2}': [reward_r2_full, judge_r2_full, mean_item_r2],
    'delta_pct': [(reward_r2_full/reward_base_full-1)*100, d_judge, d_itemmean],
}).to_csv(RESULTS_R2 / 'judge_eval_r2.csv', index=False)

print(f"Сохранено в {RESULTS_R2}")

Сохранено в ..\results\rl_r2_results


### Сравнение с R0

In [17]:
r0_full_path = RESULTS / 'full_test_reward_vs_relevance.csv'
r0_judge_path = RESULTS / 'judge_eval_full_test.csv'

if r0_full_path.exists() and r0_judge_path.exists():
    r0_full = pd.read_csv(r0_full_path)
    r0_judge = pd.read_csv(r0_judge_path)

    # Из full_test_reward_vs_relevance.csv: base и rl_beta2.0
    r0_reward = r0_full.loc[r0_full['metric'] == 'reward_top10', 'rl_beta2.0'].values[0]
    r0_ndcg   = r0_full.loc[r0_full['metric'] == 'NDCG@10', 'rl_beta2.0'].values[0]
    r0_recall = r0_full.loc[r0_full['metric'] == 'Recall@10', 'rl_beta2.0'].values[0]
    r0_map    = r0_full.loc[r0_full['metric'] == 'MAP@10', 'rl_beta2.0'].values[0]

    # Из judge_eval_full_test.csv: judge_reward_top10 и mean_item_mean_top10
    r0_judge_reward = r0_judge.loc[r0_judge['metric'] == 'judge_reward_top10', 'rl_beta2.0'].values[0]
    r0_item_mean    = r0_judge.loc[r0_judge['metric'] == 'mean_item_mean_top10', 'rl_beta2.0'].values[0]

    print(f"\n{'='*80}")
    print(f"Сравнение Base / R0 (β=2.0) / R2 (β={best_beta_r2}) на полном test")
    print(f"{'='*80}")
    print(f"{'Метрика':<24} {'Base':>10} {'R0':>10} {'R2':>10}")
    print("-" * 80)
    print(f"{'RM reward top-10':<24} {reward_base_full:>10.4f} {r0_reward:>10.4f} {reward_r2_full:>10.4f}")
    print(f"{'Judge reward top-10':<24} {judge_base_full:>10.4f} {r0_judge_reward:>10.4f} {judge_r2_full:>10.4f}")
    print(f"{'Mean item_mean top-10':<24} {mean_item_base:>10.4f} {r0_item_mean:>10.4f} {mean_item_r2:>10.4f}")
    print(f"{'NDCG@10':<24} {base_full['NDCG@10']:>10.4f} {r0_ndcg:>10.4f} {r2_full['NDCG@10']:>10.4f}")
    print(f"{'Recall@10':<24} {base_full['Recall@10']:>10.4f} {r0_recall:>10.4f} {r2_full['Recall@10']:>10.4f}")
    print(f"{'MAP@10':<24} {base_full['MAP@10']:>10.4f} {r0_map:>10.4f} {r2_full['MAP@10']:>10.4f}")
    print("-" * 80)
else:
    print("Файлы R0 не найдены — сравнение с R0 пропущено.")
    print("Ожидаемые файлы:")
    print(f"  {r0_full_path}")
    print(f"  {r0_judge_path}")


Сравнение Base / R0 (β=2.0) / R2 (β=2.0) на полном test
Метрика                        Base         R0         R2
--------------------------------------------------------------------------------
RM reward top-10             0.5092     0.5502     0.5011
Judge reward top-10          0.4817     0.5069     0.4740
Mean item_mean top-10        0.4850     0.5245     0.4731
NDCG@10                      0.1304     0.1290     0.1305
Recall@10                    0.2192     0.2179     0.2188
MAP@10                       0.1030     0.1016     0.1032
--------------------------------------------------------------------------------


*Как вывод: R2 своим центрированием убивает весь прирост R0, что показывает, что **reward в R0 растёт преимущественно из-за смещения в популярное, а не персонализации**, и сигнал персонализации в RM слишком слабый (R^2 = 0.06) для профитного использования*